<a href="https://colab.research.google.com/github/plnu-biomechanics/kin4042/blob/main/notebooks/kin4042_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://www.pointloma.edu/sites/default/files/styles/basic_page/public/images/PLNU_Biomechanics_Lab_green_yellowSD_HiRes.png" width="400">

# **KIN 4042 Sport Informatics**
**Instructor:** Arnel Aguinaldo, PhD

**Lab 2: Between-session reliability of CMJ performance and asymmetry**

In this lab, you will reproduce the principal summary and reliability analyses from:

Pérez-Castilla, A., García-Ramos, A., Janicijevic, D., Delgado-García, G., De la Cruz, J. C., Rojas, F. J., & Cepero, M. (2021). Between-session reliability of performance and asymmetry variables obtained during unilateral and bilateral countermovement jumps in basketball players. *PLOS ONE, 16*(7), e0255458. https://doi.org/10.1371/journal.pone.0255458

Twenty-three basketball players completed two identical CMJ testing sessions separated by seven days. The notebook uses the public, athlete-level supporting dataset and can be downloaded or ingested directly from our lab's GitHub repository.

*   `pone_0255458_s001.csv` from the PLNU Biomechanics Lab

To further process the data for this lab, follow the steps in this **Colab notebook**, which contains instructions and sample code on how to wrangle and analyze the data.

## Learning objectives

By the end of this lab, you should be able to:

1. import and audit repeated-measures sport-performance data;
2. calculate signed inter-limb asymmetry;
3. summarize performance and asymmetry by session;
4. choose between a paired *t* test and Wilcoxon signed-rank test;
5. calculate Cohen's *d*, SEM-based coefficient of variation (CV), and ICC(3,1);
6. interpret absolute and relative reliability together.

> **Replication note:** The public CSV contains rounded values. Small differences from the published table are expected. It also does not identify each player's preferred leg. Therefore, unilateral asymmetry can be reproduced directly, whereas the paper's preference-leg coding for bilateral asymmetry cannot be reconstructed exactly.

### Create your own Colab Notebook

1. Go to **File -> New notebook in Drive** to open a new notebook in your Python environment:<br>
<img src="https://raw.githubusercontent.com/plnu-biomechanics/kin6015/main/notebooks/images/file_notebook.png" width=450>

2. Rename your Colab notebook using this naming format: **lastname_kin4042_lab#.ipynb** (e.g., "aguinaldo_kin4042_lab1.ipynb")
3. Click on the **+ Code** option above to insert a new code cell: <br>
<img src="https://raw.githubusercontent.com/plnu-biomechanics/kin6015/main/notebooks/images/addcode.png" width=280>

4. The data you will parse and analyze for this lab will be temporarily stored in your Colab's runtime directory, which can be accessed by clicking on the folder icon in the left menu:<br>
<img src="https://raw.githubusercontent.com/plnu-biomechanics/kin6015/main/notebooks/images/colab_folder.png" width=400>

5. Copy the following lines of code to import the packages needed for this analysis and to load the data files into your working directory. **Note**: These files are "runtime" access only, meaning they are only temporarily stored in your working directory and show up when your notebook is in session.


## 1. 🔄 Import the Python libraries

### ✋ Suggested GenAI prompt — Import the analysis libraries

> I am working in Google Colab on a repeated-measures CMJ reliability lab. Generate one Python code cell that imports all the required libraries needed to perform the reliability analyses of this lab, including `pathlib`, `urllib.request`, `NumPy`, `pandas`, `Matplotlib`, `seaborn`, `scipy.stats`, sklearn's `cohen_kappa_score`, and `IPython.display.display`. Set useful pandas display options and a readable seaborn theme. Use only packages normally available in Colab and briefly explain what each library contributes.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
# Core libraries available in Google Colab; no additional installation is required.
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.metrics import cohen_kappa_score
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 3)
sns.set_theme(style="whitegrid", context="notebook")

## 2. Download and load the data

The code downloads the CSV into the temporary Colab runtime. If the GitHub URL is unavailable, upload the CSV into the same folder and rerun the loading cell.

### ✋ Suggested GenAI prompt — Create a folder and download the dataset

> Generate Python code for Google Colab that creates the folder kin4042/lab2 with pathlib, then downloads pone_0255458_s001.csv from the PLNU Biomechanics Lab GitHub raw URL into that folder using urllib.request. The code should be safe to rerun and should print the completed file path.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
data_dir = Path("kin4042/lab2")
data_dir.mkdir(parents=True, exist_ok=True)

data_url = (
    "https://raw.githubusercontent.com/plnu-biomechanics/"
    "kin4042/refs/heads/main/labs/pone_0255458_s001.csv"
)
data_path = data_dir / "pone_0255458_s001.csv"

urllib.request.urlretrieve(data_url, data_path)
print(f"Downloaded: {data_path}")

Your working directory should now look something like this:

<img src="https://raw.githubusercontent.com/plnu-biomechanics/kin4042/main/notebooks/images/kin4042_create-lab-directory.png" width=400>


### 🔄 Data Frames

<img src="https://raw.githubusercontent.com/plnu-biomechanics/kin4042/main/notebooks/images/pandas_logo.jpg" width=400>

Before the data can be analyzed, they need to be parsed into `DataFrames` as described in previous labs.

In this lab, we will use the powerful `pandas` library to create a data frame from the `pone_0255458_s001.csv` file.

You can use your favorite GenAI agent to help you with populating these two data frames.

### ✋ Suggested GenAI prompt — Load and clean the CSV

> Load the CMJ CSV into a pandas DataFrame named cmj. Read it with utf-8-sig encoding, replace hyphens in all column names with underscores, sort rows by Code and Session, reset the index, and display the shape and first five rows. Explain why utf-8-sig and standardized column names are useful.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
# utf-8-sig safely removes the byte-order mark found in the original CSV.
cmj = pd.read_csv(data_path, encoding="utf-8-sig")

# Replace hyphens in column names such as CI-UNICMJ_RIGHT.
cmj.columns = cmj.columns.str.replace("-", "_", regex=False)
cmj = cmj.sort_values(["Code", "Session"]).reset_index(drop=True)

print(f"Shape: {cmj.shape}")
display(cmj.head())

### 🧹 Data audit

### ✋ Suggested GenAI prompt — Audit the repeated-measures dataset

> Using the pandas DataFrame cmj, generate code that reports the number of rows, columns, unique players, unique sessions, missing cells, and players with two sessions. Add assertions that confirm 46 rows, 23 players, sessions 1 and 2, two sessions per player, and no missing values. Make any failed check return an informative message.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
sessions_per_player = cmj.groupby("Code")["Session"].nunique()

audit = pd.Series({
    "rows": len(cmj),
    "columns": cmj.shape[1],
    "unique_players": cmj["Code"].nunique(),
    "unique_sessions": cmj["Session"].nunique(),
    "missing_cells": int(cmj.isna().sum().sum()),
    "players_with_two_sessions": int((sessions_per_player == 2).sum()),
}, name="Value").to_frame()

display(audit)

assert len(cmj) == 46, "Expected 46 rows."
assert cmj["Code"].nunique() == 23, "Expected 23 players."
assert set(cmj["Session"].unique()) == {1, 2}, "Expected sessions 1 and 2."
assert sessions_per_player.eq(2).all(), "Every player must have two sessions."
assert cmj.isna().sum().sum() == 0, "Unexpected missing data found."

print("Data audit passed.")

## 📓Lab Note

Take note of the number of unique players with CMJ data in the data frame and the number of unique sessions per player.

## 3. Reshape and summarize the performance variables

Variable prefixes:

| Prefix | Metric | Unit |
|---|---|---|
| MF | Mean force | N |
| PF | Peak force | N |
| MV | Mean velocity | m/s |
| PV | Peak velocity | m/s |
| MP | Mean power | W |
| PP | Peak power | W |
| CI | Concentric impulse | N·s |
| JH | Jump height | m |

`UNICMJ` denotes a unilateral CMJ; `BICMJ` denotes a bilateral CMJ. The final component identifies the right or left limb.

### ✋ Suggested GenAI prompt — Reshape performance data to long format

> The DataFrame cmj has Code and Session identifiers and CMJ columns named like MF_UNICMJ_RIGHT. Generate pandas code that reshapes all performance columns to long format with Variable and Value columns. Parse Variable into Metric, Jump, and Limb using a regular expression; recode UNICMJ and BICMJ to readable labels; and display eight rows. Preserve Code and Session.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
id_columns = ["Code", "Session"]
performance_columns = [column for column in cmj.columns if column not in id_columns]

performance_long = cmj.melt(
    id_vars=id_columns,
    value_vars=performance_columns,
    var_name="Variable",
    value_name="Value",
)

components = performance_long["Variable"].str.extract(
    r"^(?P<Metric>[A-Z]+)_(?P<Jump>UNICMJ|BICMJ)_(?P<Limb>RIGHT|LEFT)$"
)
performance_long = pd.concat([performance_long, components], axis=1)
performance_long["Jump"] = performance_long["Jump"].map({
    "UNICMJ": "Unilateral CMJ",
    "BICMJ": "Bilateral CMJ",
})
performance_long["Limb"] = performance_long["Limb"].str.title()

display(performance_long.head(8))

## 📊 Data Visualization: Summary Statistics

Let's visualize the descriptive statistics for all metrics by `session` from `cmj` using box plots. This can help us see how each metric differs across sessions.

### ✋ Suggested GenAI prompt — Calculate descriptive statistics

> Using performance_long, calculate n, mean, sample standard deviation, minimum, and maximum for every Jump × Metric × Limb × Session combination. Return a tidy DataFrame named performance_summary and display a formatted pandas Styler table with two decimal places.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
performance_summary = (
    performance_long
    .groupby(["Jump", "Metric", "Limb", "Session"], as_index=False)
    .agg(
        n=("Value", "count"),
        Mean=("Value", "mean"),
        SD=("Value", "std"),
        Minimum=("Value", "min"),
        Maximum=("Value", "max"),
    )
)

display(
    performance_summary.style
    .format({"Mean": "{:.2f}", "SD": "{:.2f}", "Minimum": "{:.2f}", "Maximum": "{:.2f}"})
    .set_caption("CMJ performance by session")
)

### 📊 Visualize paired session values

### ✋ Suggested GenAI prompt — Create a paired session plot

> Generate Matplotlib code that loops through all the CMJ variables, prints each one as a PNG image, and saves a zipped file of all of the images to my working directory. Plot Session 1 and Session 2 values, connect the same player with a light line, overlay colored points, label both axes, and add a title. Use Code to group the paired observations and the cmj DataFrame as input.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
# =======================================
# Code generated or assisted by Google Gemini
# =======================================
import zipfile

output_images_dir = Path("kin4042/lab2/plots")
output_images_dir.mkdir(parents=True, exist_ok=True)

image_files = []
for variable in performance_columns:
    selected = cmj[["Code", "Session", variable]].copy()

    fig, ax = plt.subplots(figsize=(7, 5))
    for _, player in selected.groupby("Code"):
        ax.plot(player["Session"], player[variable], color="0.70", alpha=0.65, linewidth=1)
    ax.scatter(selected["Session"], selected[variable], c=selected["Session"],
               cmap="viridis", s=42, zorder=3)
    ax.set(xticks=[1, 2], xlabel="Session", ylabel=variable,
           title=f"Paired values: {variable}")

    # Save the plot as a PNG image
    image_filename = f"paired_values_{variable}.png"
    image_path = output_images_dir / image_filename
    fig.savefig(image_path)
    image_files.append(image_path)
    plt.close(fig) # Close the figure to free up memory

# Create a zip file containing all the images
zip_filename = output_images_dir / "cmj_paired_plots.zip"
with zipfile.ZipFile(zip_filename, "w") as zf:
    for file_path in image_files:
        zf.write(file_path, arcname=file_path.name)

print(f"All CMJ paired plots saved to: {zip_filename}")
print(f"Individual plots saved in: {output_images_dir}/")

## 📓Lab Note

Download the zip file that contain all of your plots so that you can include them in your lab write-up. Organize them as you see fit.




## 4. 📊 Calculate inter-limb asymmetry

For unilateral CMJs, the article used the percentage-difference formula

$$
100\left(1-\frac{\min(R,L)}{\max(R,L)}\right).
$$

We add a sign so that positive scores indicate a larger right-leg value and negative scores indicate a larger left-leg value:

$$
\text{signed unilateral asymmetry}=100\frac{R-L}{\max(R,L)}.
$$

### ✋ Suggested GenAI prompt — Calculate inter-limb asymmetry variables

> Create vectorized Python functions for two inter-limb asymmetry calculations. For unilateral CMJ, calculate signed percentage difference as 100*(right-left)/max(right,left), where positive favors right. For bilateral CMJ, calculate unsigned magnitude as 100*abs(right-left)/(right+left). Return NaN when the denominator is zero. Apply the functions to MF, PF, MV, PV, MP, PP, and CI; also calculate unilateral JH asymmetry. Add the new columns to cmj_analysis.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
def signed_unilateral_asymmetry(right, left):
    # Signed percentage difference: positive favors right, negative favors left.
    denominator = np.maximum(right, left)
    return np.where(denominator == 0, np.nan, 100 * (right - left) / denominator)


def bilateral_asymmetry_magnitude(right, left):
    # Unsigned bilateral asymmetry magnitude; does not require preferred leg.
    denominator = right + left
    return np.where(denominator == 0, np.nan, 100 * np.abs(right - left) / denominator)


metric_codes = ["MF", "PF", "MV", "PV", "MP", "PP", "CI"]
cmj_analysis = cmj.copy()

for metric in metric_codes:
    cmj_analysis[f"ASYM_{metric}_UNICMJ"] = signed_unilateral_asymmetry(
        cmj_analysis[f"{metric}_UNICMJ_RIGHT"],
        cmj_analysis[f"{metric}_UNICMJ_LEFT"],
    )
    cmj_analysis[f"ASYM_{metric}_BICMJ_MAG"] = bilateral_asymmetry_magnitude(
        cmj_analysis[f"{metric}_BICMJ_RIGHT"],
        cmj_analysis[f"{metric}_BICMJ_LEFT"],
    )

# Jump height was reported only for unilateral CMJs.
cmj_analysis["ASYM_JH_UNICMJ"] = signed_unilateral_asymmetry(
    cmj_analysis["JH_UNICMJ_RIGHT"],
    cmj_analysis["JH_UNICMJ_LEFT"],
)

unilateral_asymmetry_columns = [
    column for column in cmj_analysis.columns
    if column.startswith("ASYM_") and column.endswith("_UNICMJ")
]
bilateral_magnitude_columns = [
    column for column in cmj_analysis.columns
    if column.startswith("ASYM_") and column.endswith("_BICMJ_MAG")
]

display(cmj_analysis[["Code", "Session"] + unilateral_asymmetry_columns].head())

### ✋ Suggested GenAI prompt — Summarize the asymmetry variables

> Reshape the asymmetry columns in cmj_analysis to a long DataFrame with Code, Session, Variable, and Asymmetry. Label unilateral signed asymmetry separately from the bilateral magnitude extension, extract the metric code, and calculate n, mean, sample SD, minimum, and maximum by analysis, metric, and session. Display a formatted table with two decimals.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
asymmetry_long = cmj_analysis.melt(
    id_vars=["Code", "Session"],
    value_vars=unilateral_asymmetry_columns + bilateral_magnitude_columns,
    var_name="Variable",
    value_name="Asymmetry",
)
asymmetry_long["Analysis"] = np.where(
    asymmetry_long["Variable"].str.endswith("_BICMJ_MAG"),
    "Bilateral magnitude (extension)",
    "Unilateral signed asymmetry",
)
asymmetry_long["Metric"] = asymmetry_long["Variable"].str.extract(r"^ASYM_([A-Z]+)_")

asymmetry_summary = (
    asymmetry_long
    .groupby(["Analysis", "Metric", "Session"], as_index=False)
    .agg(
        n=("Asymmetry", "count"),
        Mean=("Asymmetry", "mean"),
        SD=("Asymmetry", "std"),
        Minimum=("Asymmetry", "min"),
        Maximum=("Asymmetry", "max"),
    )
)

display(
    asymmetry_summary.style
    .format({"Mean": "{:.2f}", "SD": "{:.2f}", "Minimum": "{:.2f}", "Maximum": "{:.2f}"})
    .set_caption("Inter-limb asymmetry by session")
)

## 5. 📊 Reliability-analysis functions

For each variable, we will calculate:

- a paired *t* test when paired differences are normally distributed, or a Wilcoxon signed-rank test otherwise;
- Cohen's *d* using $(M_2-M_1)/SD_{pooled}$, matching the article's definition;
- SEM and CV from the residual error of a two-way subject-by-session ANOVA;
- ICC(3,1): two-way mixed-effects, consistency, single measure;
- 95% confidence intervals for CV and ICC.

The article classified a performance metric as acceptable when **ICC > 0.70 and CV < 10%**. CV is not used for signed asymmetry because its mean can approach zero.

### ✋ Suggested GenAI prompt — Build reusable reliability functions

> Write reusable Python functions for a two-session reliability analysis. First, pivot one variable into complete, player-aligned Session 1 and Session 2 arrays. Then calculate ICC(3,1), single-measure model using ANOVA mean squares and an F-based 95% CI. Finally, create a wrapper that tests the normality of paired differences, selects a paired *t* test or Wilcoxon signed-rank test, calculates pooled-SD Cohen's d, SEM, CV=100*SEM/mean, a chi-square 95% CI for CV, and returns all results as a dictionary. Allow CV to be omitted for signed asymmetry.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
def make_pairs(data, variable):
    # Return complete Session 1 and Session 2 arrays aligned by player.
    paired = (
        data[["Code", "Session", variable]]
        .pivot(index="Code", columns="Session", values=variable)
        .dropna(subset=[1, 2])
        .sort_index()
    )
    return paired[1].to_numpy(float), paired[2].to_numpy(float)


def icc_3_1(session_1, session_2, alpha=0.05):
    # ICC(3,1): two-way mixed, consistency, single-measure ICC.
    values = np.column_stack([session_1, session_2])
    n, k = values.shape
    grand_mean = values.mean()
    subject_means = values.mean(axis=1)
    session_means = values.mean(axis=0)

    ss_subject = k * np.sum((subject_means - grand_mean) ** 2)
    residuals = values - subject_means[:, None] - session_means[None, :] + grand_mean
    ss_error = np.sum(residuals ** 2)

    df_subject = n - 1
    df_error = (n - 1) * (k - 1)
    ms_subject = ss_subject / df_subject
    ms_error = ss_error / df_error

    icc = (ms_subject - ms_error) / (ms_subject + (k - 1) * ms_error)

    # F-based confidence interval for ICC(C,1). Spreadsheet/package conventions
    # can produce slightly different limits, but the point estimate is identical.
    f_ratio = ms_subject / ms_error
    f_lower = f_ratio / stats.f.ppf(1 - alpha / 2, df_subject, df_error)
    f_upper = f_ratio * stats.f.ppf(1 - alpha / 2, df_error, df_subject)
    icc_low = (f_lower - 1) / (f_lower + k - 1)
    icc_high = (f_upper - 1) / (f_upper + k - 1)

    return {
        "ICC_3_1": icc,
        "ICC_low": icc_low,
        "ICC_high": icc_high,
        "MS_error": ms_error,
        "df_error": df_error,
    }


def analyze_reliability(data, variable, report_cv=True):
    session_1, session_2 = make_pairs(data, variable)
    difference = session_2 - session_1
    n = len(difference)

    shapiro = stats.shapiro(difference)
    if shapiro.pvalue > 0.05:
        comparison = stats.ttest_rel(session_2, session_1)
        test_name = "Paired t test"
    else:
        comparison = stats.wilcoxon(session_2, session_1, method="approx", correction=False)
        test_name = "Wilcoxon signed-rank"

    pooled_sd = np.sqrt((np.var(session_1, ddof=1) + np.var(session_2, ddof=1)) / 2)
    cohens_d = (np.mean(session_2) - np.mean(session_1)) / pooled_sd

    icc = icc_3_1(session_1, session_2)
    sem = np.sqrt(icc["MS_error"])

    if report_cv:
        mean_score = np.mean(np.concatenate([session_1, session_2]))
        cv = 100 * sem / mean_score
        df_error = icc["df_error"]
        cv_low = cv * np.sqrt(df_error / stats.chi2.ppf(0.975, df_error))
        cv_high = cv * np.sqrt(df_error / stats.chi2.ppf(0.025, df_error))
    else:
        sem = cv = cv_low = cv_high = np.nan

    return {
        "Variable": variable,
        "n": n,
        "Session_1_Mean": np.mean(session_1),
        "Session_1_SD": np.std(session_1, ddof=1),
        "Session_2_Mean": np.mean(session_2),
        "Session_2_SD": np.std(session_2, ddof=1),
        "Test": test_name,
        "Normality_p": shapiro.pvalue,
        "Comparison_p": comparison.pvalue,
        "Cohens_d": cohens_d,
        "SEM": sem,
        "CV": cv,
        "CV_low": cv_low,
        "CV_high": cv_high,
        "ICC_3_1": icc["ICC_3_1"],
        "ICC_low": icc["ICC_low"],
        "ICC_high": icc["ICC_high"],
    }

## 6. 📈 Check normality of paired differences

A paired *t* test assumes that the **within-player differences** are normally distributed. It does not require each session by itself to be normal.

### ✋ Suggested GenAI prompt — Test normality of paired differences

> For every performance variable and unilateral asymmetry variable in cmj_analysis, align Session 1 and Session 2 by Code and run a Shapiro-Wilk test on Session 2 minus Session 1. Create a table containing Variable, n, W, p, and a recommended paired t test when p>.05 or Wilcoxon signed-rank test otherwise. Explain why the paired differences—not each session separately—are tested.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
analysis_variables = performance_columns + unilateral_asymmetry_columns

normality_results = []
for variable in analysis_variables:
    session_1, session_2 = make_pairs(cmj_analysis, variable)
    result = stats.shapiro(session_2 - session_1)
    normality_results.append({
        "Variable": variable,
        "n": len(session_1),
        "W": result.statistic,
        "p": result.pvalue,
        "Recommended_test": "Paired t test" if result.pvalue > 0.05 else "Wilcoxon signed-rank",
    })

normality_results = pd.DataFrame(normality_results)
display(
    normality_results.style
    .format({"W": "{:.3f}", "p": "{:.3f}"})
    .set_caption("Shapiro-Wilk tests of paired differences")
)

## 7. 📊 Performance reliability

### ✋ Suggested GenAI prompt — Analyze performance reliability

> Apply my analyze_reliability function to every CMJ performance column. Add Jump, Limb, and Metric labels parsed from each column name. Classify reliability as Acceptable only when ICC(3,1)>.70 and CV<10%; otherwise label it Review. Display session means and SDs, test, p, Cohen's d, CV with 95% CI, ICC with 95% CI, and the classification in a formatted table. Highlight rows needing review.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
performance_reliability = pd.DataFrame([
    analyze_reliability(cmj_analysis, variable, report_cv=True)
    for variable in performance_columns
])

performance_reliability["Jump"] = np.select(
    [
        performance_reliability["Variable"].str.contains("_UNICMJ_"),
        performance_reliability["Variable"].str.contains("_BICMJ_"),
    ],
    ["Unilateral CMJ", "Bilateral CMJ"],
    default="Unknown",
)
performance_reliability["Limb"] = np.where(
    performance_reliability["Variable"].str.endswith("_RIGHT"), "Right", "Left"
)
performance_reliability["Metric"] = performance_reliability["Variable"].str.extract(r"^([A-Z]+)")
performance_reliability["Reliability"] = np.where(
    (performance_reliability["ICC_3_1"] > 0.70) & (performance_reliability["CV"] < 10),
    "Acceptable",
    "Review",
)

performance_table = performance_reliability[[
    "Jump", "Metric", "Limb", "n",
    "Session_1_Mean", "Session_1_SD", "Session_2_Mean", "Session_2_SD",
    "Test", "Comparison_p", "Cohens_d", "CV", "CV_low", "CV_high",
    "ICC_3_1", "ICC_low", "ICC_high", "Reliability",
]].copy()

display(
    performance_table.style
    .format({
        "Session_1_Mean": "{:.2f}", "Session_1_SD": "{:.2f}",
        "Session_2_Mean": "{:.2f}", "Session_2_SD": "{:.2f}",
        "Comparison_p": "{:.3f}", "Cohens_d": "{:.2f}",
        "CV": "{:.2f}", "CV_low": "{:.2f}", "CV_high": "{:.2f}",
        "ICC_3_1": "{:.2f}", "ICC_low": "{:.2f}", "ICC_high": "{:.2f}",
    })
    .map(lambda value: "background-color: #fff2cc" if value == "Review" else "",
         subset=["Reliability"])
    .set_caption("Between-session reliability of CMJ performance")
)

## 8. 📊 Unilateral asymmetry reliability

The paper reported ICC, but not CV, for asymmetry. CV would be unstable when the signed group mean is close to zero.

### ✋ Suggested GenAI prompt — Analyze unilateral asymmetry reliability

> Apply my analyze_reliability function to all signed unilateral asymmetry columns, with CV disabled because signed means may approach zero. Extract the metric code, classify ICC values above .70 as acceptable, and display session means and SDs, selected paired test, p, Cohen's d, ICC(3,1), and its 95% CI. Highlight unacceptable ICCs.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
asymmetry_reliability = pd.DataFrame([
    analyze_reliability(cmj_analysis, variable, report_cv=False)
    for variable in unilateral_asymmetry_columns
])
asymmetry_reliability["Metric"] = asymmetry_reliability["Variable"].str.extract(
    r"^ASYM_([A-Z]+)_"
)
asymmetry_reliability["ICC_interpretation"] = np.where(
    asymmetry_reliability["ICC_3_1"] > 0.70, "Acceptable", "Unacceptable"
)

asymmetry_table = asymmetry_reliability[[
    "Metric", "n", "Session_1_Mean", "Session_1_SD", "Session_2_Mean",
    "Session_2_SD", "Test", "Comparison_p", "Cohens_d",
    "ICC_3_1", "ICC_low", "ICC_high", "ICC_interpretation",
]]

display(
    asymmetry_table.style
    .format({
        "Session_1_Mean": "{:.2f}", "Session_1_SD": "{:.2f}",
        "Session_2_Mean": "{:.2f}", "Session_2_SD": "{:.2f}",
        "Comparison_p": "{:.3f}", "Cohens_d": "{:.2f}",
        "ICC_3_1": "{:.2f}", "ICC_low": "{:.2f}", "ICC_high": "{:.2f}",
    })
    .map(lambda value: "background-color: #fff2cc" if value == "Unacceptable" else "",
         subset=["ICC_interpretation"])
    .set_caption("Between-session reliability of unilateral CMJ asymmetry")
)

### Direction-of-asymmetry agreement

ICC evaluates the consistency of asymmetry **scores**. Cohen's kappa evaluates whether the same side was favored in both sessions. The article interpreted kappa as poor (≤0), slight (.01–.20), fair (.21–.40), moderate (.41–.60), substantial (.61–.80), or almost perfect (.81–.99).

### ✋ Suggested GenAI prompt — Evaluate direction-of-asymmetry agreement

> For each signed unilateral asymmetry variable, code nonnegative scores as Right and negative scores as Left for each session. Calculate Cohen's kappa between sessions, count how many players favor the same side twice, and classify kappa as poor, slight, fair, moderate, substantial, or almost perfect using the Landis and Koch ranges supplied in the lab. Return and display a tidy results table.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
def interpret_kappa(value):
    if value <= 0.00:
        return "Poor"
    if value <= 0.20:
        return "Slight"
    if value <= 0.40:
        return "Fair"
    if value <= 0.60:
        return "Moderate"
    if value <= 0.80:
        return "Substantial"
    return "Almost perfect"


kappa_results = []
for variable in unilateral_asymmetry_columns:
    session_1, session_2 = make_pairs(cmj_analysis, variable)
    direction_1 = np.where(session_1 >= 0, "Right", "Left")
    direction_2 = np.where(session_2 >= 0, "Right", "Left")
    kappa = cohen_kappa_score(direction_1, direction_2)
    kappa_results.append({
        "Metric": variable.split("_")[1],
        "Kappa": kappa,
        "Agreement": interpret_kappa(kappa),
        "Same_direction_n": int(np.sum(direction_1 == direction_2)),
        "Total_n": len(direction_1),
    })

kappa_results = pd.DataFrame(kappa_results)
display(
    kappa_results.style
    .format({"Kappa": "{:.2f}"})
    .set_caption("Agreement in the direction of unilateral CMJ asymmetry")
)

## 9. 📊 Verification checks

### ✋ Suggested GenAI prompt — Add verification checks

> Write Python assertions that confirm the calculated public-CSV results for MF_UNICMJ_RIGHT are approximately ICC(3,1)=0.87350, CV=5.33608%, and paired-test p=.22122, and that MF_UNICMJ_LEFT is approximately ICC=.91879 and CV=4.44766%. Use np.isclose with a sensible tolerance and print a success message only when all checks pass.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
# These checks verify the analysis against values independently calculated from
# the public CSV. They prevent silent errors if the code is modified.
check_right = performance_reliability.loc[
    performance_reliability["Variable"] == "MF_UNICMJ_RIGHT"
].iloc[0]
check_left = performance_reliability.loc[
    performance_reliability["Variable"] == "MF_UNICMJ_LEFT"
].iloc[0]

assert np.isclose(check_right["ICC_3_1"], 0.87350, atol=1e-4)
assert np.isclose(check_right["CV"], 5.33608, atol=1e-4)
assert np.isclose(check_right["Comparison_p"], 0.22122, atol=1e-4)
assert np.isclose(check_left["ICC_3_1"], 0.91879, atol=1e-4)
assert np.isclose(check_left["CV"], 4.44766, atol=1e-4)

print("Verification checks passed.")

## 10. 📓 Interpretation questions

1. Why should ICC and CV be interpreted together rather than separately?
2. Which performance variables met both criteria for acceptable reliability?
3. Did jump height show the same reliability as force, velocity, power, or impulse?
4. Which unilateral asymmetry metric had the highest ICC? Did any exceed .70?
5. Why is CV misleading for a signed variable with a mean close to zero?
6. What does a non-significant paired comparison tell you? Why does it **not** prove reliability?
7. Compare your results with Table 2 of Pérez-Castilla et al. (2021). How might rounding, normality-test decisions, and confidence-interval conventions explain differences?

## 11. Save the analysis tables

### ✋ Suggested GenAI prompt — Export the analysis tables

> Create a kin4042/lab2/outputs folder that is safe to recreate. Export the performance summary, asymmetry summary, performance reliability, unilateral asymmetry reliability, and unilateral kappa DataFrames as CSV files without row indices. Copy and compress all CSV files into a single zipped file. Finally, print the names of every CSV and the zipped file saved in the folder.

After generating the code, compare it with the guide cell below, run it, and confirm that the output is reasonable. Add a comment naming the agent that assisted you, for example:

```python
# =======================================
# Code generated or assisted by [agent]
# =======================================
```

In [ ]:
output_dir = data_dir / "outputs"
output_dir.mkdir(exist_ok=True)

performance_summary.to_csv(output_dir / "performance_summary.csv", index=False)
asymmetry_summary.to_csv(output_dir / "asymmetry_summary.csv", index=False)
performance_reliability.to_csv(output_dir / "performance_reliability.csv", index=False)
asymmetry_reliability.to_csv(output_dir / "unilateral_asymmetry_reliability.csv", index=False)
kappa_results.to_csv(output_dir / "unilateral_asymmetry_kappa.csv", index=False)

print("Saved files:")
for path in sorted(output_dir.glob("*.csv")):
    print(f"- {path}")

# Create a zip file containing all the images
zip_filename = output_dir / "cmj_summary_tables.zip"
with zipfile.ZipFile(zip_filename, "w") as zf:
    for file_path in sorted(output_dir.glob("*.csv")):
        zf.write(file_path, arcname=file_path.name)

print(f"Saved zip file: {zip_filename}")

## 📓Lab Note

Download and use incorporate these tables into your lab write-up. Format them in APA and cite them in the text of your write-up

## 💡 Final note

Reliability is multidimensional. A metric can show a high ICC because players maintain their rank order, yet still have enough within-player noise to produce an unacceptable CV. Conversely, the absence of a significant session difference does not establish reliability. Interpret systematic change, standardized effect size, absolute error, relative reliability, and direction-of-asymmetry agreement together.